# Model Complexity and Capacity

**Topic:** Learning Capacity and Model Size

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what model capacity means and how it relates to the number of parameters
- **Explain** why increasing capacity always reduces training error but has a sweet spot for test error
- **Interpret** how regularization constrains capacity without changing model architecture

> **Tip:** Set capacity (max depth) to 1 and increase slowly. Watch how training accuracy climbs toward 100% while validation accuracy peaks and then either plateaus or falls. The peak is the optimal capacity for this dataset.

---
## How we got here

In **11_the_curse_of_dimensionality.ipynb** we saw that more features can hurt. Model complexity follows a parallel logic: more capacity can also hurt, unless it is constrained.

This connects directly to:
- **math/calculus/09_optimization.ipynb** — you explored loss landscapes and local minima, which become more complex as model capacity grows
- **math/statistics_for_ml/04_regularization_mathematics.ipynb** — you derived L1 and L2 regularization, which impose constraints on parameter magnitude

Here we see those constraints in action on learning curves.

---
## Why this matters for data science

Capacity is the most important dial you control when designing a model. Too little and the model cannot learn. Too much and it memorizes. Finding the sweet spot, and knowing how to use regularization to shift that sweet spot, is a core skill.

More complex models also have practical costs: they are slower to train, harder to interpret, and more sensitive to hyperparameter choices. The right capacity is the smallest one that achieves acceptable validation performance.

---
## Try it yourself

In [2]:
from plotly.subplots import make_subplots
from IPython.display import HTML

np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=2, n_informative=2,
    n_redundant=0, flip_y=0.20, class_sep=0.8, random_state=42)

# Fixed boundary mesh — computed once
_pad = 0.5
_xs  = np.linspace(X[:, 0].min() - _pad, X[:, 0].max() + _pad, 150)
_ys  = np.linspace(X[:, 1].min() - _pad, X[:, 1].max() + _pad, 150)
_XX, _YY = np.meshgrid(_xs, _ys)
_grid = np.c_[_XX.ravel(), _YY.ravel()]

# Curve cache — recomputed only when toggle changes, never on slider move
_curve_cache = {}

def get_curve(regularize):
    if regularize in _curve_cache:
        return _curve_cache[regularize]
    depths = list(range(1, 21))
    tr, va = [], []
    for d in depths:
        kw = {"max_depth": d, "random_state": 42}
        if regularize:
            kw["min_samples_leaf"] = 10
        clf = DecisionTreeClassifier(**kw).fit(X, y)
        tr.append(clf.score(X, y))
        va.append(cross_val_score(clf, X, y, cv=5).mean())
    _curve_cache[regularize] = (depths, tr, va)
    return _curve_cache[regularize]

ACCENT = PALETTE.get("accent", PALETTE["secondary"])
_BOUNDARY_CS = [[0, "rgba(76,110,245,0.18)"], [1, "rgba(247,103,7,0.18)"]]

out = Output()
_initialized = False

capacity_slider = IntSlider(value=3, min=1, max=20, step=1,
    description="Max depth:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))
reg_toggle = ToggleButtons(
    options=["Unconstrained", "Regularized (min_samples_leaf=10)"],
    value="Unconstrained",
    layout=widgets.Layout(width="440px"))

def render(max_d, regularize):
    global _initialized
    depths, tr, va = get_curve(regularize)
    peak_d = depths[int(np.argmax(va))]

    # Fit boundary tree at current depth only
    kw = {"max_depth": max_d, "random_state": 42}
    if regularize:
        kw["min_samples_leaf"] = 10
    _clf = DecisionTreeClassifier(**kw).fit(X, y)
    Z = _clf.predict(_grid).reshape(_XX.shape).astype(float)

    fig = make_subplots(rows=1, cols=2, column_widths=[0.52, 0.48],
        subplot_titles=[f"Bias–variance curve  (depth = {max_d})",
                        f"Decision boundary at depth {max_d}"])

    # LEFT: bias-variance curves
    fig.add_trace(go.Scatter(x=depths, y=tr, mode="lines+markers",
        line=dict(color=ACCENT, width=2),
        marker=dict(size=5), name="Training accuracy"), row=1, col=1)
    fig.add_trace(go.Scatter(x=depths, y=va, mode="lines+markers",
        line=dict(color=PALETTE["secondary"], width=2),
        marker=dict(size=5), name="Validation accuracy"), row=1, col=1)
    fig.add_trace(go.Scatter(x=[max_d], y=[va[max_d - 1]], mode="markers",
        marker=dict(size=14, color=PALETTE["primary"],
                    line=dict(color="white", width=2)),
        name="Current depth"), row=1, col=1)
    fig.update_xaxes(title_text="Max tree depth", row=1, col=1)
    fig.update_yaxes(title_text="Accuracy", range=[0.4, 1.02], row=1, col=1)

    # RIGHT: decision boundary heatmap + class scatter
    fig.add_trace(go.Heatmap(x=_xs, y=_ys, z=Z,
        colorscale=_BOUNDARY_CS, showscale=False, opacity=1.0), row=1, col=2)
    for cls, color in [(0, PALETTE["primary"]), (1, PALETTE["secondary"])]:
        mask = y == cls
        fig.add_trace(go.Scatter(
            x=X[mask, 0], y=X[mask, 1], mode="markers",
            marker=dict(color=color, size=5, opacity=0.8,
                        line=dict(color="white", width=0.5)),
            name=f"Class {cls}"), row=1, col=2)
    fig.update_xaxes(showticklabels=False, row=1, col=2)
    fig.update_yaxes(showticklabels=False, row=1, col=2)

    suffix = " (regularized)" if regularize else ""
    fig.update_layout(
        title=dict(
            text=(f"depth {max_d}{suffix}  |  "
                  f"train {tr[max_d-1]:.2f}  val {va[max_d-1]:.2f}"),
            font=dict(size=FONT["size_title"], family=FONT["family"])),
        paper_bgcolor=PALETTE["background"],
        height=430,
        margin=dict(t=75, b=30, l=50, r=20),
        showlegend=True)

    # Interpretation
    if max_d < peak_d:
        regime = ("Underfitting: the boundary is too simple to separate the classes. "
                  "Both training and validation accuracy are still rising.")
    elif max_d <= peak_d + 1:
        regime = (f"Near the sweet spot: validation accuracy is at its highest "
                  f"(~{max(va):.2f} around depth {peak_d}). This is the capacity to pick.")
    else:
        regime = (f"Overfitting: training accuracy is {tr[max_d-1]:.2f} but validation has "
                  f"fallen to {va[max_d-1]:.2f}. The boundary is carving out individual "
                  f"noisy points it won't see again.")

    gap_line = (f"Generalization gap = train − val = {tr[max_d-1] - va[max_d-1]:.2f}. "
                f"A widening gap as depth grows is the signature of memorization.")

    if not regularize:
        toggle_line = ("Unconstrained: the tree keeps splitting until leaves are pure, "
                       "so training accuracy reaches 1.00.")
    else:
        toggle_line = ("min_samples_leaf=10 caps capacity: each leaf must hold ≥10 points, "
                       "so the boundary stays smooth and training tops out below 1.00 — "
                       "the same depth overfits less.")

    interp = (
        f'<div style="font-family:{FONT["family"]};font-size:13px;background:#f8f9fa;'
        f'padding:14px 18px;border-radius:6px;margin-top:8px;line-height:1.65;color:#333;">'
        f'<p style="margin:0 0 6px 0;">{regime}</p>'
        f'<p style="margin:0 0 6px 0;">{gap_line}</p>'
        f'<p style="margin:0;">{toggle_line}</p>'
        f'</div>'
    )

    with out:
        clear_output(wait=True)
        fig.show()
        display(HTML(interp))
    _initialized = True

def on_change(change):
    render(capacity_slider.value, reg_toggle.value.startswith("Regularized"))

capacity_slider.observe(on_change, names="value")
reg_toggle.observe(on_change, names="value")

display(VBox([capacity_slider, reg_toggle, out]))
render(3, False)

---
## What's happening?

Think of model capacity like a filing cabinet. A cabinet with two drawers can store limited information — it might miss important details (underfitting). A cabinet with 200 drawers can store everything including irrelevant noise — it takes longer to search and the noise clutters the useful content (overfitting).

For a decision tree, capacity = maximum depth:
- Depth 1: one split, two possible predictions. Very limited.
- Depth 5-7: enough splits to capture complex patterns.
- Depth 20+: enough splits to memorize every training example individually.

Regularization constrains capacity. `min_samples_leaf=10` forces each leaf to represent at least 10 examples — it cannot memorize individual points.

| Model | Capacity | Parameters | Flexibility | Risk of overfitting |
|---|---|---|---|---|
| Linear Regression | Low | n_features + 1 | Low | Low |
| Decision Tree (depth 5) | Medium | ~31 nodes | Medium | Medium |
| Decision Tree (depth 20) | Very high | ~1M nodes | Very high | Very high |
| Random Forest (100 trees) | High | ~100× depth-5 | High | Managed by averaging |
| Neural Network (4 layers) | Very high | Millions | Extreme | Requires regularization |

---
## Real-world example: Neural network for image classification

ResNet-50, a standard image classifier, has 25 million parameters. Trained on ImageNet's 1.2 million images, it generalizes well. Trained on 1,000 images, it memorizes completely.

Capacity must be matched to dataset size. For small datasets, start with a shallow model and add capacity only if the validation error is still high (underfitting). The filing cabinet should have exactly as many drawers as your information requires, no more.

---
## Key takeaway

> **More capacity always reduces training error, but test error has a sweet spot — and regularization lets you constrain capacity without changing the model architecture.**

---
*Next up: Interpretability vs Complexity — the tradeoff that decides which models are appropriate in which industries*